## Balanced Variational-Equilibrium Execution Profile

This upgraded version preserves the practical public SciQ execution profile while making **variational equilibrium** the central scientific layer. The notebook still evaluates retrieval, calibration, selective prediction, risk strata, evidence stability, and bootstrap comparisons, but now adds an explicit equilibrium functional, equilibrium residual, force-balance diagnostics, Pareto-style operating analysis, and equilibrium-oriented figures/tables.


# VERITA-QA: Variational Equilibrium Retrieval Intelligence for Trustworthy Answering

This notebook upgrades the earlier UVIF retrieval question-answering workflow into **VERITA-QA** (**V**ariational **E**quilibrium **R**etrieval **I**ntelligence for **T**rustworthy **A**nswering). The aim is not to claim that the proposed UVIF layer always maximizes raw accuracy, but to model retrieval-augmented question answering as a **controllable variational-equilibrium system**.

The notebook treats each model output as an operating point that must balance predictive utility, calibration risk, epistemic uncertainty, evidence redundancy, evidence coverage, evidence complexity, and selective-action efficiency.

Default dataset: **SciQ** from Hugging Face (`allenai/sciq`).

Scientific emphasis:
- evidence selection as variational energy minimization,
- model comparison through equilibrium rather than accuracy alone,
- calibration and uncertainty as stabilizing forces,
- residual imbalance as a measurable risk signal,
- dynamic decision-field trajectories rather than only static scalar scores,
- controllable equilibrium as the distinctive VERITA-QA contribution.


## Variational-equilibrium formulation

The upgraded notebook interprets a QA decision as an equilibrium state rather than a single maximum-score prediction. For each model or policy, an operating point is characterized by several competing forces:

\[
\mathcal{F}_{VE}
= -\lambda_U U
+ \lambda_C C_{cal}
+ \lambda_H H
+ \lambda_R R_{red}
+ \lambda_K K
- \lambda_G G_{cov}
- \lambda_A A_{sel}.
\]

Here, lower \(\mathcal{F}_{VE}\) indicates a better equilibrium. The terms denote: predictive utility \(U\), calibration error \(C_{cal}\), normalized uncertainty \(H\), evidence redundancy \(R_{red}\), complexity \(K\), evidence coverage \(G_{cov}\), and selective-action efficiency \(A_{sel}\). The interpretation is simple: a reliable QA system should not only answer correctly; it should also remain calibrated, avoid unnecessary redundant evidence, keep uncertainty under control, and preserve useful coverage.

The notebook also computes an **equilibrium residual**:

\[
\rho_{VE} = \left| \mathcal{F}_{VE} - \min_m \mathcal{F}_{VE}^{(m)} \right|,
\]

which measures how far each operating point remains from the most stable observed equilibrium among the tested model variants.


In [ ]:
# ============================================================
# Cell 1 — Environment setup
# ============================================================
import sys, subprocess, importlib, os, time, json, math, random, warnings
warnings.filterwarnings('ignore')

REQUIRED = ['datasets', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'tqdm']

def ensure_package(pkg):
    module = pkg.replace('-', '_')
    if pkg == 'scikit-learn':
        module = 'sklearn'
    try:
        importlib.import_module(module)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in REQUIRED:
    ensure_package(pkg)

print('Environment ready.')

In [ ]:
# ============================================================
# Cell 2 — Imports and output directories
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, f1_score

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Outputs/VERITA_QA_Variational_Equilibrium')
else:
    BASE_DIR = Path.cwd() / 'VERITA_QA_Variational_Equilibrium'

FIG_DIR    = BASE_DIR / 'Figures'
TABLE_DIR  = BASE_DIR / 'Tables'
OUTPUT_DIR = BASE_DIR / 'Outputs'
LOG_DIR    = BASE_DIR / 'logs'
PYTHON_DIR = BASE_DIR / 'Notebook'
for d in [FIG_DIR, TABLE_DIR, OUTPUT_DIR, LOG_DIR, PYTHON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)

In [ ]:
# ============================================================
# Cell 3 — Balanced variational-equilibrium configuration
# ============================================================
CONFIG = {
    'dataset_name': 'sciq',
    'max_train_questions': 3000,
    'max_eval_questions': 500,
    'max_corpus_passages': 7000,
    'random_seed': 42,

    # Retrieval parameters reduced for speed
    'top_k_dense': 20,
    'top_k_sparse': 20,
    'rerank_budget': 15,
    'context_k': 4,
    'rrf_k0': 60,
    'dense_dim': 128,

    # UVIF tuning reduced for speed
    'max_uvif_tune_questions': 50,
    'uvif_temperature': 10.0,
    'uvif_grid': {
        'w_dense': [0.50],
        'w_sparse': [1.00, 1.25],
        'lambda_utility': [1.00],
        'lambda_risk': [0.00, 0.10],
        'lambda_redundancy': [0.00, 0.05],
        'lambda_coverage': [0.10]
    },

    # Trustworthiness analyses
    'n_bootstrap': 100,
    'calibration_bins': 10,
    'selective_points': 10,

    # Fast validation-based calibration
    'temperature_grid': [0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 5.00, 8.00],
    'calibration_metric': 'ece',
    'run_optional_arc': False,
}

SEED = CONFIG['random_seed']
random.seed(SEED)
np.random.seed(SEED)
CONFIG

In [ ]:
# ============================================================
# Cell 3b — Variational-equilibrium weights and reporting options
# ============================================================
VE_CONFIG = {
    # Lower total free-energy is better. These weights can be tuned later.
    'lambda_utility': 1.00,
    'lambda_calibration': 0.90,
    'lambda_uncertainty': 0.65,
    'lambda_redundancy': 0.45,
    'lambda_complexity': 0.30,
    'lambda_coverage': 0.55,
    'lambda_action': 0.50,

    # Selective prediction coverage used as an operational-action term.
    'action_coverage_reference': 0.80,

    # Numerical stability.
    'eps': 1e-12,
}

VE_CONFIG


DYNAMIC_CONFIG = {
    'n_steps': 25,
    'dt': 0.20,
    'alpha_relaxation': 0.85,   # relaxation toward target operating point
    'beta_coupling': 0.20,      # force-balance coupling between stabilizing and destabilizing axes
    'gamma_utility': 0.08,      # utility injection into stabilizing dynamics
    'delta_risk': 0.08,         # risk damping in destabilizing dynamics
    'trajectory_seed': 2026,
}

DYNAMIC_CONFIG


In [ ]:
# ============================================================
# Cell 4 — Load SciQ public dataset
# ============================================================
def normalize_text(x):
    return ' '.join(str(x).replace('\\n', ' ').replace('\\r', ' ').split())

def load_sciq():
    ds = load_dataset('allenai/sciq')
    def convert(row, split):
        opts = [row['distractor1'], row['distractor2'], row['distractor3'], row['correct_answer']]
        rng = random.Random(abs(hash(row['question'])) % (2**32))
        rng.shuffle(opts)
        return {
            'id': f'{split}_{abs(hash(row["question"]))}',
            'question': normalize_text(row['question']),
            'options': [normalize_text(o) for o in opts],
            'answer_idx': opts.index(row['correct_answer']),
            'answer': normalize_text(row['correct_answer']),
            'support': normalize_text(row.get('support', '') or ''),
            'source': 'SciQ'
        }
    return {split: [convert(r, split) for r in ds[split]] for split in ds.keys()}

data = load_sciq()
train_rows = data['train'][:CONFIG['max_train_questions']]
val_rows = data['validation'][:CONFIG['max_eval_questions']]
test_rows = data['test'][:CONFIG['max_eval_questions']]

print('Train/Val/Test:', len(train_rows), len(val_rows), len(test_rows))
print(test_rows[0])

In [ ]:
# ============================================================
# Cell 5 — Build retrieval corpus from public supports
# ============================================================
all_rows = train_rows + val_rows + test_rows
seen = set()
supports = []
row_support_to_pid = {}

for r in all_rows:
    s = normalize_text(r.get('support', ''))
    if len(s.split()) >= 6:
        if s not in seen:
            seen.add(s)
            supports.append({'passage_id': f'p{len(supports)}', 'text': s, 'source': r['source']})
        row_support_to_pid[r['id']] = list(seen).index(s) if False else None

# Map text to pid after corpus creation
support_text_to_pid = {x['text']: i for i, x in enumerate(supports)}
for r in all_rows:
    s = normalize_text(r.get('support', ''))
    r['support_pid'] = support_text_to_pid.get(s, None)

supports = supports[:CONFIG['max_corpus_passages']]
corpus_df = pd.DataFrame(supports)
corpus_texts = corpus_df['text'].tolist()
corpus_df.to_csv(TABLE_DIR / 'public_corpus_passages.csv', index=False)

stats = pd.DataFrame([
    {'Split': 'Training', 'Questions': len(train_rows)},
    {'Split': 'Validation', 'Questions': len(val_rows)},
    {'Split': 'Test', 'Questions': len(test_rows)},
    {'Split': 'Retrieval corpus', 'Questions': len(corpus_df)}
])
stats.to_csv(TABLE_DIR / 'Table_1_public_dataset_statistics.csv', index=False)
print('Corpus passages:', len(corpus_df))
stats

In [ ]:
# ============================================================
# Cell 6 — Option-aware query construction
# ============================================================
def build_query(row, option_aware=True):
    q = normalize_text(row['question'])
    if not option_aware:
        return q
    labels = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    parts = [q]
    for i, opt in enumerate(row['options']):
        parts.append(f'[SEP] Option {labels[i]}: {normalize_text(opt)}')
    return ' '.join(parts)

print(build_query(train_rows[0]))

In [ ]:
# ============================================================
# Cell 7 — Build sparse and dense retrieval indexes
# ============================================================
sparse_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=1, max_features=50000)
X_sparse = sparse_vectorizer.fit_transform(corpus_texts)

svd_dim = min(CONFIG['dense_dim'], max(2, min(X_sparse.shape) - 1))
svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
X_dense = normalize(svd.fit_transform(X_sparse))

print('Sparse matrix:', X_sparse.shape)
print('Dense proxy matrix:', X_dense.shape)

In [ ]:
# ============================================================
# Cell 8 — Fast retrieval utilities with cached candidates
# ============================================================
def topk_from_scores(scores, k):
    scores = np.asarray(scores).ravel()
    k = min(k, len(scores))
    if k <= 0:
        return []
    idx = np.argpartition(-scores, k-1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]

def retrieve_candidates(row, option_aware=True):
    query = build_query(row, option_aware=option_aware)
    q_sparse = sparse_vectorizer.transform([query])
    sparse_scores = (q_sparse @ X_sparse.T).toarray()[0]
    sparse_top = topk_from_scores(sparse_scores, CONFIG['top_k_sparse'])

    q_dense = normalize(svd.transform(q_sparse))
    dense_scores = (q_dense @ X_dense.T).ravel()
    dense_top = topk_from_scores(dense_scores, CONFIG['top_k_dense'])

    # RRF fusion with both dense and sparse ranks.
    ranks = {}
    for rank, (idx, sc) in enumerate(sparse_top, 1):
        ranks.setdefault(idx, {})['sparse_rank'] = rank
        ranks[idx]['sparse_score'] = sc
    for rank, (idx, sc) in enumerate(dense_top, 1):
        ranks.setdefault(idx, {})['dense_rank'] = rank
        ranks[idx]['dense_score'] = sc

    max_rank = max(CONFIG['top_k_sparse'], CONFIG['top_k_dense']) + 1
    fused = []
    for idx, d in ranks.items():
        sr = d.get('sparse_rank', max_rank)
        dr = d.get('dense_rank', max_rank)
        rrf = 1/(CONFIG['rrf_k0'] + sr) + 1/(CONFIG['rrf_k0'] + dr)
        fused.append({
            'pid': idx,
            'sparse_score': d.get('sparse_score', 0.0),
            'dense_score': d.get('dense_score', 0.0),
            'rrf_score': rrf,
            'text': corpus_texts[idx]
        })
    fused = sorted(fused, key=lambda x: x['rrf_score'], reverse=True)[:CONFIG['rerank_budget']]
    return fused

def lexical_coverage(question, passage):
    q_terms = set([t.lower() for t in question.split() if len(t) > 3])
    p_terms = set([t.lower().strip('.,;:!?()[]') for t in passage.split()])
    if not q_terms:
        return 0.0
    return len(q_terms & p_terms) / len(q_terms)

def redundancy_penalty(pid, selected):
    if not selected:
        return 0.0
    v = X_dense[pid]
    sims = [float(np.dot(v, X_dense[s])) for s in selected]
    return max(sims) if sims else 0.0

EVAL_ROWS = test_rows
TUNE_ROWS = val_rows[:CONFIG['max_uvif_tune_questions']]
ALL_RUN_ROWS = TUNE_ROWS + EVAL_ROWS
candidate_cache = {r['id']: retrieve_candidates(r, option_aware=True) for r in tqdm(ALL_RUN_ROWS, desc='Caching candidates')}
print('Cached candidate sets:', len(candidate_cache))

In [ ]:
# ============================================================
# Cell 9 — Option scoring and model variants
# ============================================================
def softmax(x, temperature=1.0):
    x = np.asarray(x, dtype=float) / max(temperature, 1e-8)
    x = x - np.max(x)
    e = np.exp(x)
    return e / max(e.sum(), 1e-12)

def option_scores(row, context_ids, alpha_support=1.0):
    # Scores each candidate option against selected evidence.
    context = ' '.join([corpus_texts[i] for i in context_ids]) if context_ids else row['question']
    option_statements = [row['question'] + ' ' + opt for opt in row['options']]
    docs = [context] + option_statements
    local_vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=1)
    M = local_vec.fit_transform(docs)
    sims = (M[1:] @ M[0].T).toarray().ravel()

    # Lightweight lexical boost: option tokens appearing in context.
    ctx_terms = set(context.lower().split())
    boosts = []
    for opt in row['options']:
        terms = [t.lower() for t in opt.split() if len(t) > 2]
        boosts.append(sum(t in ctx_terms for t in terms) / max(1, len(terms)))
    return sims + alpha_support * np.asarray(boosts)

def predict_with_context(row, context_ids, temperature=1.0):
    scores = option_scores(row, context_ids)
    probs = softmax(scores, temperature=temperature)
    pred = int(np.argmax(probs))
    confidence = float(np.max(probs))
    entropy = float(-np.sum(probs * np.log(probs + 1e-12)) / np.log(len(probs)))
    margin = float(np.sort(probs)[-1] - np.sort(probs)[-2]) if len(probs) > 1 else confidence
    return pred, confidence, entropy, margin, scores, probs


def apply_temperature_to_df(df, temperature):
    """Recompute confidence, entropy, margin, and probabilities from stored option scores.
    Positive temperature scaling preserves the predicted class but changes calibration.
    """
    out = df.copy()
    probs_list, confs, ents, margins = [], [], [], []
    for scores in out['scores']:
        p = softmax(scores, temperature=temperature)
        probs_list.append(p.tolist())
        confs.append(float(np.max(p)))
        ents.append(float(-np.sum(p * np.log(p + 1e-12)) / np.log(len(p))))
        margins.append(float(np.sort(p)[-1] - np.sort(p)[-2]) if len(p) > 1 else float(np.max(p)))
    out['probs'] = probs_list
    out['confidence'] = confs
    out['entropy'] = ents
    out['margin'] = margins
    return out

def mean_nll_from_records(df, temperature=1.0):
    losses = []
    for _, r in df.iterrows():
        p = softmax(r['scores'], temperature=temperature)
        losses.append(-np.log(p[int(r['gold'])] + 1e-12))
    return float(np.mean(losses))

def tune_temperature_from_validation(df_val, grid=None, metric='ece'):
    if grid is None:
        grid = CONFIG['temperature_grid']
    rows = []
    best_t, best_score = None, float('inf')
    for t in grid:
        tmp = apply_temperature_to_df(df_val, t)
        ece, _ = ece_from_records(tmp, CONFIG['calibration_bins'])
        nll = mean_nll_from_records(df_val, temperature=t)
        score = ece if metric == 'ece' else nll
        rows.append({'temperature': t, 'val_ece': ece, 'val_nll': nll, 'selection_score': score})
        if score < best_score:
            best_score, best_t = score, t
    return float(best_t), pd.DataFrame(rows)

def context_for_model(row, mode, uvif_params=None):
    cands = candidate_cache[row['id']]
    if mode == 'no_rag':
        return []
    if mode == 'sparse':
        return [c['pid'] for c in sorted(cands, key=lambda x: x['sparse_score'], reverse=True)[:CONFIG['context_k']]]
    if mode == 'dense':
        return [c['pid'] for c in sorted(cands, key=lambda x: x['dense_score'], reverse=True)[:CONFIG['context_k']]]
    if mode == 'hyrec':
        # Cross-encoder proxy: combine fused rank with query coverage.
        scored = []
        for c in cands:
            ce = c['rrf_score'] + 0.25 * lexical_coverage(row['question'], c['text'])
            scored.append((c['pid'], ce))
        return [pid for pid, _ in sorted(scored, key=lambda x: x[1], reverse=True)[:CONFIG['context_k']]]
    if mode == 'uvif':
        return uvif_select_context(row, uvif_params)
    raise ValueError(mode)

def evaluate_rows(rows, mode, uvif_params=None, temperature=1.0):
    recs = []
    for r in rows:
        ctx = context_for_model(r, mode, uvif_params=uvif_params)
        pred, conf, ent, margin, scores, probs = predict_with_context(r, ctx, temperature=temperature)
        recs.append({
            'id': r['id'], 'pred': pred, 'gold': r['answer_idx'], 'correct': int(pred == r['answer_idx']),
            'confidence': conf, 'entropy': ent, 'margin': margin, 'context_ids': ctx,
            'scores': scores.tolist(), 'probs': probs.tolist()
        })
    df = pd.DataFrame(recs)
    return df

In [ ]:
# ============================================================
# Cell 10 — UVIF variational evidence selection
# ============================================================
def uvif_candidate_energy(row, cand, selected, params):
    # Lower energy is better.
    utility = params['w_sparse'] * cand['sparse_score'] + params['w_dense'] * cand['dense_score'] + cand['rrf_score']
    coverage = lexical_coverage(row['question'], cand['text'])
    redundancy = redundancy_penalty(cand['pid'], selected)

    # Risk proxy: low utility and high redundancy increase the energy.
    energy = (
        - params['lambda_utility'] * utility
        - params['lambda_coverage'] * coverage
        + params['lambda_redundancy'] * redundancy
    )
    return float(energy)

def uvif_select_context(row, params):
    cands = candidate_cache[row['id']]
    selected = []
    remaining = list(cands)
    while remaining and len(selected) < CONFIG['context_k']:
        scored = [(uvif_candidate_energy(row, c, selected, params), c) for c in remaining]
        scored.sort(key=lambda x: x[0])
        chosen = scored[0][1]
        selected.append(chosen['pid'])
        remaining = [c for c in remaining if c['pid'] != chosen['pid']]
    return selected

def grid_dicts(grid):
    import itertools
    keys = list(grid.keys())
    for vals in itertools.product(*[grid[k] for k in keys]):
        yield dict(zip(keys, vals))

def ece_from_records(df, n_bins=10):
    conf = df['confidence'].to_numpy(float)
    corr = df['correct'].to_numpy(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf >= lo) & (conf < hi if i < n_bins-1 else conf <= hi)
        if mask.sum() == 0:
            rows.append({'bin_low': lo, 'bin_high': hi, 'count': 0, 'accuracy': np.nan, 'confidence': np.nan})
            continue
        acc = corr[mask].mean()
        avg = conf[mask].mean()
        ece += (mask.sum() / len(df)) * abs(acc - avg)
        rows.append({'bin_low': lo, 'bin_high': hi, 'count': int(mask.sum()), 'accuracy': acc, 'confidence': avg})
    return float(ece), pd.DataFrame(rows)

def summarize_records(df):
    acc = accuracy_score(df['gold'], df['pred'])
    f1 = f1_score(df['gold'], df['pred'], average='macro')
    ece, _ = ece_from_records(df, CONFIG['calibration_bins'])
    return acc, f1, ece

# Validation tuning uses accuracy first, ECE second. Compact grid keeps runtime low.
search_rows = []
best_params, best_tuple = None, (-1, 999)
for params in tqdm(list(grid_dicts(CONFIG['uvif_grid'])), desc='Fast UVIF grid'):
    df = evaluate_rows(TUNE_ROWS, mode='uvif', uvif_params=params)
    acc, f1, ece = summarize_records(df)
    search_rows.append({**params, 'val_accuracy': acc, 'val_macro_f1': f1, 'val_ece': ece})
    key = (acc, -ece)
    if key > best_tuple:
        best_tuple = key
        best_params = params

BEST_UVIF_PARAMS = best_params
uvif_search_df = pd.DataFrame(search_rows).sort_values(['val_accuracy', 'val_ece'], ascending=[False, True])
uvif_search_df.to_csv(TABLE_DIR / 'Table_UVIF_validation_weight_search.csv', index=False)
pd.DataFrame([BEST_UVIF_PARAMS]).to_csv(TABLE_DIR / 'Table_UVIF_selected_weights.csv', index=False)
print('Selected UVIF params:', BEST_UVIF_PARAMS)
uvif_search_df.head(10)

In [ ]:
# ============================================================
# Cell 11 — Held-out test evaluation with validation temperature scaling
# ============================================================
model_specs = [
    ('w/o RAG', 'no_rag', None),
    ('Sparse Only', 'sparse', None),
    ('Dense Only', 'dense', None),
    ('Hybrid w/o Rerank', 'hyrec', None),
    ('VERITA-QA', 'uvif', BEST_UVIF_PARAMS),
]

# 1) Evaluate validation/tuning rows once per model to select calibration temperature.
validation_results = {}
temperature_rows = []
temperature_search_tables = []
for name, mode, params in tqdm(model_specs, desc='Tuning calibration temperatures'):
    val_df = evaluate_rows(TUNE_ROWS, mode=mode, uvif_params=params, temperature=1.0)
    validation_results[name] = val_df
    best_t, tsearch = tune_temperature_from_validation(
        val_df,
        grid=CONFIG['temperature_grid'],
        metric=CONFIG['calibration_metric']
    )
    tsearch['Model'] = name
    temperature_search_tables.append(tsearch)
    temperature_rows.append({'Model': name, 'Selected temperature': best_t})

temperature_df = pd.DataFrame(temperature_rows)
temperature_search_df = pd.concat(temperature_search_tables, ignore_index=True)
temperature_df.to_csv(TABLE_DIR / 'Table_2a_public_selected_temperatures.csv', index=False)
temperature_search_df.to_csv(TABLE_DIR / 'Table_2b_public_temperature_search.csv', index=False)

# 2) Evaluate held-out test rows once; then apply each selected temperature.
results = {}
calibrated_results = {}
perf_rows = []
cal_rows = []

for name, mode, params in tqdm(model_specs, desc='Evaluating models'):
    raw_df = evaluate_rows(EVAL_ROWS, mode=mode, uvif_params=params, temperature=1.0)
    temp = float(temperature_df.loc[temperature_df['Model'] == name, 'Selected temperature'].iloc[0])
    cal_df = apply_temperature_to_df(raw_df, temp)

    acc_raw, f1_raw, ece_raw = summarize_records(raw_df)
    acc_cal, f1_cal, ece_cal = summarize_records(cal_df)

    results[name] = raw_df
    calibrated_results[name] = cal_df

    perf_rows.append({'Model': name, 'Accuracy': acc_raw, 'Macro-F1': f1_raw, 'ECE': ece_raw})
    cal_rows.append({
        'Model': name,
        'Accuracy': acc_raw,
        'Macro-F1': f1_raw,
        'ECE_before': ece_raw,
        'Selected_temperature': temp,
        'ECE_after': ece_cal,
        'Delta_ECE_after_minus_before': ece_cal - ece_raw,
        'Relative_ECE_reduction_%': 100 * (ece_raw - ece_cal) / max(ece_raw, 1e-12)
    })

performance_df = pd.DataFrame(perf_rows)
calibration_comparison_df = pd.DataFrame(cal_rows)

performance_df.to_csv(TABLE_DIR / 'Table_2_public_overall_performance_uncalibrated.csv', index=False)
calibration_comparison_df.to_csv(TABLE_DIR / 'Table_2c_public_pre_post_calibration.csv', index=False)

print('Uncalibrated performance:')
display(performance_df)
print('Pre/post temperature scaling calibration:')
display(calibration_comparison_df)


In [ ]:
# ============================================================
# Cell 12 — Retrieval quality and evidence stability
# ============================================================
def relevant_pid(row):
    pid = row.get('support_pid', None)
    if pid is None or (isinstance(pid, float) and np.isnan(pid)):
        return None
    return int(pid) if int(pid) < len(corpus_texts) else None

def retrieval_metrics_for_variant(rows, variant):
    hits5, hits10, rr = [], [], []
    for r in rows:
        rel = relevant_pid(r)
        if rel is None:
            continue
        if variant == 'Sparse Only':
            ranked = [c['pid'] for c in sorted(candidate_cache[r['id']], key=lambda x: x['sparse_score'], reverse=True)]
        elif variant == 'Dense Only':
            ranked = [c['pid'] for c in sorted(candidate_cache[r['id']], key=lambda x: x['dense_score'], reverse=True)]
        elif variant == 'Hybrid w/o Rerank':
            ranked = [c['pid'] for c in sorted(candidate_cache[r['id']], key=lambda x: x['rrf_score'], reverse=True)]
        else:
            ranked = context_for_model(r, 'uvif', BEST_UVIF_PARAMS)
        hits5.append(int(rel in ranked[:5]))
        hits10.append(int(rel in ranked[:10]))
        rr.append(1/(ranked.index(rel)+1) if rel in ranked else 0.0)
    return np.mean(hits5), np.mean(hits10), np.mean(rr)

ret_rows = []
for variant in ['Sparse Only', 'Dense Only', 'Hybrid w/o Rerank', 'VERITA-QA']:
    r5, r10, mrr = retrieval_metrics_for_variant(EVAL_ROWS, variant)
    ret_rows.append({'Variant': variant, 'R@5': r5, 'R@10': r10, 'MRR': mrr})
retrieval_df = pd.DataFrame(ret_rows)
retrieval_df.to_csv(TABLE_DIR / 'Table_3_public_retrieval_quality.csv', index=False)

# Evidence stability: redundancy and coverage of selected context.
def context_redundancy(ctx):
    if len(ctx) <= 1:
        return 0.0
    sims = []
    for i in range(len(ctx)):
        for j in range(i+1, len(ctx)):
            sims.append(float(np.dot(X_dense[ctx[i]], X_dense[ctx[j]])))
    return float(np.mean(sims)) if sims else 0.0

stab_rows = []
for name, mode, params in [('Sparse Only','sparse',None), ('Hybrid w/o Rerank','hyrec',None), ('VERITA-QA','uvif',BEST_UVIF_PARAMS)]:
    reds, covs = [], []
    for r in EVAL_ROWS:
        ctx = context_for_model(r, mode, params)
        reds.append(context_redundancy(ctx))
        covs.append(np.mean([lexical_coverage(r['question'], corpus_texts[p]) for p in ctx]) if ctx else 0.0)
    stab_rows.append({'Model': name, 'Avg context redundancy': np.mean(reds), 'Avg question coverage': np.mean(covs)})
stability_df = pd.DataFrame(stab_rows)
stability_df.to_csv(TABLE_DIR / 'Table_4_public_evidence_stability.csv', index=False)

display(retrieval_df)
display(stability_df)

In [ ]:
# ============================================================
# Cell 13 — Trustworthiness analyses after calibration: selective prediction and risk strata
# ============================================================
trust_rows = []
selective_rows = []
risk_rows = []
fixed_coverage_rows = []

# These analyses use calibrated confidence values. Accuracy is unchanged by temperature scaling,
# but confidence ranking, ECE, and abstention behavior become more defensible.
analysis_results = calibrated_results

for model_name, df in analysis_results.items():
    # Selective prediction: keep most confident examples only.
    df2 = df.sort_values('confidence', ascending=False).reset_index(drop=True)
    n = len(df2)
    for cov in np.linspace(0.1, 1.0, CONFIG['selective_points']):
        k = max(1, int(round(cov * n)))
        sub = df2.iloc[:k]
        ece, _ = ece_from_records(sub, CONFIG['calibration_bins'])
        selective_rows.append({
            'Model': model_name,
            'Coverage': k/n,
            'Accuracy': sub['correct'].mean(),
            'ECE': ece,
            'Abstained_fraction': 1 - k/n
        })

    # Fixed coverage points are easier to cite in the manuscript.
    for cov in [0.50, 0.70, 0.90, 1.00]:
        k = max(1, int(round(cov * n)))
        sub = df2.iloc[:k]
        ece, _ = ece_from_records(sub, CONFIG['calibration_bins'])
        fixed_coverage_rows.append({
            'Model': model_name,
            'Coverage_target': cov,
            'Coverage': k/n,
            'Retained_questions': k,
            'Accuracy': sub['correct'].mean(),
            'ECE': ece,
            'Mean_confidence': sub['confidence'].mean()
        })

    # Risk strata: low margin = high ambiguity risk.
    q1, q2 = df['margin'].quantile([1/3, 2/3])
    def stratum(m):
        if m <= q1: return 'High risk / low margin'
        if m <= q2: return 'Medium risk'
        return 'Low risk / high margin'
    tmp = df.copy()
    tmp['Risk stratum'] = tmp['margin'].apply(stratum)
    for s, sub in tmp.groupby('Risk stratum'):
        ece, _ = ece_from_records(sub, CONFIG['calibration_bins'])
        risk_rows.append({
            'Model': model_name,
            'Risk stratum': s,
            'Count': len(sub),
            'Accuracy': sub['correct'].mean(),
            'ECE': ece,
            'Mean confidence': sub['confidence'].mean(),
            'Mean entropy': sub['entropy'].mean()
        })

selective_df = pd.DataFrame(selective_rows)
fixed_coverage_df = pd.DataFrame(fixed_coverage_rows)
risk_df = pd.DataFrame(risk_rows)

selective_df.to_csv(TABLE_DIR / 'Table_5_public_selective_prediction_calibrated.csv', index=False)
fixed_coverage_df.to_csv(TABLE_DIR / 'Table_5b_public_fixed_coverage_reliability.csv', index=False)
risk_df.to_csv(TABLE_DIR / 'Table_6_public_risk_stratified_results_calibrated.csv', index=False)

display(selective_df.head())
display(fixed_coverage_df.head())
display(risk_df.head())


In [ ]:
# ============================================================
# Cell 14 — Calibration bins and bootstrap significance
# ============================================================
# Calibration bins are computed after validation-based temperature scaling.
calibration_tables = []
for model_name, df in calibrated_results.items():
    ece, bins = ece_from_records(df, CONFIG['calibration_bins'])
    bins['Model'] = model_name
    calibration_tables.append(bins)
calibration_df = pd.concat(calibration_tables, ignore_index=True)
calibration_df.to_csv(TABLE_DIR / 'Table_7_public_calibration_bins_after_temperature_scaling.csv', index=False)

# Also store before/after calibration bins for the strongest baseline and UVIF.
base_candidates = performance_df[performance_df['Model'] != 'VERITA-QA'].sort_values('Accuracy', ascending=False)
strongest = base_candidates.iloc[0]['Model']
prepost_bins = []
for model_name in [strongest, 'VERITA-QA']:
    for stage, source in [('Before calibration', results), ('After calibration', calibrated_results)]:
        ece, bins = ece_from_records(source[model_name], CONFIG['calibration_bins'])
        bins['Model'] = model_name
        bins['Stage'] = stage
        bins['ECE'] = ece
        prepost_bins.append(bins)
prepost_calibration_bins_df = pd.concat(prepost_bins, ignore_index=True)
prepost_calibration_bins_df.to_csv(TABLE_DIR / 'Table_7b_public_prepost_calibration_bins.csv', index=False)

# Bootstrap: UVIF against strongest accuracy baseline. Calibration does not change accuracy,
# so the bootstrap remains a predictive-performance comparison rather than a calibration test.
uvif = results['VERITA-QA']['correct'].to_numpy()
base = results[strongest]['correct'].to_numpy()

diffs = []
rng = np.random.default_rng(SEED)
for _ in range(CONFIG['n_bootstrap']):
    idx = rng.integers(0, len(uvif), len(uvif))
    diffs.append(float(uvif[idx].mean() - base[idx].mean()))
diffs = np.asarray(diffs)
mean_diff = float(diffs.mean())
lo, hi = np.percentile(diffs, [2.5, 97.5])
p_two = float(2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0)))
p_two = max(1e-4, min(1.0, p_two))
boot_df = pd.DataFrame([{
    'Comparison': f'VERITA-QA vs {strongest}',
    'mean_accuracy_diff': mean_diff,
    'ci95_low': lo,
    'ci95_high': hi,
    'p_two_sided': p_two
}])
boot_df.to_csv(TABLE_DIR / 'Table_8_public_bootstrap_significance.csv', index=False)

display(calibration_df.head())
display(prepost_calibration_bins_df.head())
display(boot_df)


# Numerical safety for reporting
boot_df['p_two_sided'] = boot_df['p_two_sided'].clip(lower=1e-4)


In [ ]:
# ============================================================
# Cell 14b — Variational-equilibrium analysis
# ============================================================
def _safe_model_value(df, model, column, default=np.nan):
    sub = df[df['Model'] == model]
    if len(sub) == 0 or column not in sub.columns:
        return default
    return float(sub.iloc[0][column])

def _normalize_series(values, higher_is_better=True):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    out = np.zeros_like(arr, dtype=float)
    if finite.sum() == 0:
        return out
    lo, hi = np.nanmin(arr[finite]), np.nanmax(arr[finite])
    if abs(hi - lo) < 1e-12:
        out[finite] = 0.5
    else:
        out[finite] = (arr[finite] - lo) / (hi - lo)
    if not higher_is_better:
        out = 1.0 - out
    out[~finite] = np.nan
    return out

def variational_equilibrium_table():
    rows = []
    # Selective action term: performance at a fixed operational coverage, if available.
    ref_cov = VE_CONFIG['action_coverage_reference']
    for model_name, df in calibrated_results.items():
        acc = _safe_model_value(performance_df, model_name, 'Accuracy')
        ece_after = _safe_model_value(calibration_comparison_df, model_name, 'ECE_after')
        entropy = float(df['entropy'].mean())
        margin = float(df['margin'].mean())
        redundancy = _safe_model_value(stability_df, model_name, 'Avg context redundancy', default=0.0)
        coverage = _safe_model_value(stability_df, model_name, 'Avg question coverage', default=0.0)
        context_complexity = float(np.mean([len(x) for x in df['context_ids']])) if 'context_ids' in df else 0.0
        # Robust fixed-coverage lookup. Earlier notebook versions stored only the realized
        # coverage column ('Coverage'), while upgraded versions may also store the requested
        # target column ('Coverage_target'). We support both to avoid KeyError.
        fc_sub = fixed_coverage_df[fixed_coverage_df['Model'] == model_name].copy()
        if len(fc_sub):
            if 'Coverage_target' in fc_sub.columns:
                fc_sub['_coverage_distance'] = (fc_sub['Coverage_target'].astype(float) - ref_cov).abs()
            elif 'Coverage' in fc_sub.columns:
                fc_sub['_coverage_distance'] = (fc_sub['Coverage'].astype(float) - ref_cov).abs()
            else:
                fc_sub['_coverage_distance'] = np.inf
            fc_sub = fc_sub.sort_values('_coverage_distance')
            action_acc = float(fc_sub.iloc[0]['Accuracy']) if 'Accuracy' in fc_sub.columns else acc
        else:
            action_acc = acc
        rows.append({
            'Model': model_name,
            'Predictive utility U': acc,
            'Calibration risk C_cal': ece_after,
            'Uncertainty H': entropy,
            'Decision margin': margin,
            'Evidence redundancy R_red': redundancy,
            'Evidence coverage G_cov': coverage,
            'Evidence complexity K': context_complexity,
            'Selective-action utility A_sel': action_acc,
        })
    ve = pd.DataFrame(rows)

    # Normalize into comparable force coordinates.
    ve['U_norm'] = _normalize_series(ve['Predictive utility U'], higher_is_better=True)
    ve['C_norm'] = _normalize_series(ve['Calibration risk C_cal'], higher_is_better=False)  # higher means safer
    ve['H_norm'] = _normalize_series(ve['Uncertainty H'], higher_is_better=False)
    ve['R_norm'] = _normalize_series(ve['Evidence redundancy R_red'], higher_is_better=False)
    ve['K_norm'] = _normalize_series(ve['Evidence complexity K'], higher_is_better=False)
    ve['G_norm'] = _normalize_series(ve['Evidence coverage G_cov'], higher_is_better=True)
    ve['A_norm'] = _normalize_series(ve['Selective-action utility A_sel'], higher_is_better=True)

    # Stabilizing force: desirable normalized properties.
    ve['Stabilizing force'] = (
        VE_CONFIG['lambda_utility'] * ve['U_norm'] +
        VE_CONFIG['lambda_calibration'] * ve['C_norm'] +
        VE_CONFIG['lambda_uncertainty'] * ve['H_norm'] +
        VE_CONFIG['lambda_redundancy'] * ve['R_norm'] +
        VE_CONFIG['lambda_complexity'] * ve['K_norm'] +
        VE_CONFIG['lambda_coverage'] * ve['G_norm'] +
        VE_CONFIG['lambda_action'] * ve['A_norm']
    )

    # Destabilizing force: observed risk burden.
    ve['Destabilizing force'] = (
        VE_CONFIG['lambda_calibration'] * (1 - ve['C_norm']) +
        VE_CONFIG['lambda_uncertainty'] * (1 - ve['H_norm']) +
        VE_CONFIG['lambda_redundancy'] * (1 - ve['R_norm']) +
        VE_CONFIG['lambda_complexity'] * (1 - ve['K_norm'])
    )

    # Free energy: lower is better.
    ve['VE free energy'] = (
        -VE_CONFIG['lambda_utility'] * ve['U_norm']
        + VE_CONFIG['lambda_calibration'] * (1 - ve['C_norm'])
        + VE_CONFIG['lambda_uncertainty'] * (1 - ve['H_norm'])
        + VE_CONFIG['lambda_redundancy'] * (1 - ve['R_norm'])
        + VE_CONFIG['lambda_complexity'] * (1 - ve['K_norm'])
        - VE_CONFIG['lambda_coverage'] * ve['G_norm']
        - VE_CONFIG['lambda_action'] * ve['A_norm']
    )
    ve['Equilibrium residual'] = ve['VE free energy'] - ve['VE free energy'].min()
    ve['Force-balance gap'] = abs(ve['Stabilizing force'] - ve['Destabilizing force'])
    ve['VE rank'] = ve['VE free energy'].rank(method='min', ascending=True).astype(int)
    return ve.sort_values('VE free energy').reset_index(drop=True)

variational_equilibrium_df = variational_equilibrium_table()
variational_equilibrium_df.to_csv(TABLE_DIR / 'Table_8b_variational_equilibrium_operating_points.csv', index=False)

# Per-sample residual for UVIF: high residual examples are useful for manuscript error analysis.
uvif_records = calibrated_results['VERITA-QA'].copy()
uvif_records['sample_free_energy'] = (
    -uvif_records['confidence']
    + VE_CONFIG['lambda_uncertainty'] * uvif_records['entropy']
    - VE_CONFIG['lambda_action'] * uvif_records['margin']
)
uvif_records['sample_equilibrium_residual'] = uvif_records['sample_free_energy'] - uvif_records['sample_free_energy'].min()
uvif_records = uvif_records.sort_values('sample_equilibrium_residual', ascending=False)
uvif_equilibrium_cases = []
id_to_row = {r['id']: r for r in EVAL_ROWS}
for _, rec in uvif_records.head(10).iterrows():
    r = id_to_row[rec['id']]
    uvif_equilibrium_cases.append({
        'Question': r['question'],
        'Correct answer': r['options'][r['answer_idx']],
        'Predicted answer': r['options'][int(rec['pred'])],
        'Correct': int(rec['correct']),
        'Confidence': float(rec['confidence']),
        'Entropy': float(rec['entropy']),
        'Margin': float(rec['margin']),
        'Sample free energy': float(rec['sample_free_energy']),
        'Sample equilibrium residual': float(rec['sample_equilibrium_residual']),
    })
uvif_equilibrium_cases_df = pd.DataFrame(uvif_equilibrium_cases)
uvif_equilibrium_cases_df.to_csv(TABLE_DIR / 'Table_8c_uvif_high_residual_cases.csv', index=False)

display(variational_equilibrium_df)
display(uvif_equilibrium_cases_df.head())


In [ ]:
# ============================================================
# Cell 15 — Figures
# ============================================================
def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', path)

# Figure 1: Accuracy-ECE tradeoff before and after calibration.
plt.figure(figsize=(8,5))
for _, r in calibration_comparison_df.iterrows():
    plt.scatter(r['ECE_before'], r['Accuracy'], s=60, marker='o')
    plt.scatter(r['ECE_after'], r['Accuracy'], s=80, marker='x')
    plt.plot([r['ECE_before'], r['ECE_after']], [r['Accuracy'], r['Accuracy']], alpha=0.4)
    plt.text(r['ECE_after']+0.002, r['Accuracy'], r['Model'], fontsize=8)
plt.xlabel('Expected Calibration Error (lower is better)')
plt.ylabel('Accuracy (higher is better)')
plt.title('Accuracy--Calibration Tradeoff Before/After Temperature Scaling')
plt.grid(alpha=0.3)
savefig('Fig1_Accuracy_Calibration_Tradeoff_PrePost.png')

# Figure 2: Overall uncalibrated metrics.
x = np.arange(len(performance_df))
plt.figure(figsize=(10,5))
plt.plot(x, performance_df['Accuracy'], marker='o', label='Accuracy')
plt.plot(x, performance_df['Macro-F1'], marker='s', label='Macro-F1')
plt.plot(x, performance_df['ECE'], marker='^', label='ECE before calibration')
plt.xticks(x, performance_df['Model'], rotation=25, ha='right')
plt.ylabel('Score')
plt.title('Fast Public Benchmark Results')
plt.legend(); plt.grid(alpha=0.3)
savefig('Fig2_Public_Performance_Analysis.png')

# Figure 3: ECE before vs after calibration.
plt.figure(figsize=(9,5))
x = np.arange(len(calibration_comparison_df))
plt.bar(x - 0.18, calibration_comparison_df['ECE_before'], width=0.36, label='Before calibration')
plt.bar(x + 0.18, calibration_comparison_df['ECE_after'], width=0.36, label='After temperature scaling')
plt.xticks(x, calibration_comparison_df['Model'], rotation=25, ha='right')
plt.ylabel('ECE')
plt.title('Validation-Based Temperature Scaling Reduces Calibration Error')
plt.legend(); plt.grid(axis='y', alpha=0.3)
savefig('Fig3_PrePost_Calibration_ECE.png')

# Figure 4: Selective prediction after calibration.
plt.figure(figsize=(8,5))
for model_name in ['Sparse Only', 'Hybrid w/o Rerank', 'VERITA-QA']:
    sub = selective_df[selective_df['Model'] == model_name]
    plt.plot(sub['Coverage'], sub['Accuracy'], marker='o', label=model_name)
plt.xlabel('Coverage retained')
plt.ylabel('Accuracy')
plt.title('Calibrated Selective Prediction: Accuracy vs Coverage')
plt.legend(); plt.grid(alpha=0.3)
savefig('Fig4_Selective_Prediction_Calibrated.png')

# Figure 5: Reliability diagram for UVIF and strongest baseline after calibration.
plt.figure(figsize=(7,5))
for model_name in [strongest, 'VERITA-QA']:
    sub = calibration_df[calibration_df['Model'] == model_name].dropna()
    centers = (sub['bin_low'] + sub['bin_high']) / 2
    plt.plot(centers, sub['accuracy'], marker='o', label=f'{model_name}')
plt.plot([0,1], [0,1], linestyle='--', label='Perfect calibration')
plt.xlabel('Confidence bin')
plt.ylabel('Empirical accuracy')
plt.title('Reliability Diagram After Temperature Scaling')
plt.legend(); plt.grid(alpha=0.3)
savefig('Fig5_Reliability_Diagram_After_Calibration.png')

# Figure 6: Evidence stability.
plt.figure(figsize=(8,5))
x = np.arange(len(stability_df))
plt.bar(x - 0.18, stability_df['Avg context redundancy'], width=0.36, label='Redundancy')
plt.bar(x + 0.18, stability_df['Avg question coverage'], width=0.36, label='Coverage')
plt.xticks(x, stability_df['Model'], rotation=20, ha='right')
plt.ylabel('Average score')
plt.title('Evidence Stability and Coverage')
plt.legend(); plt.grid(axis='y', alpha=0.3)
savefig('Fig6_Evidence_Stability.png')

# Figure 7: Risk-stratified accuracy after calibration.
plt.figure(figsize=(9,5))
plot_df = risk_df[risk_df['Model'].isin(['Sparse Only', 'Hybrid w/o Rerank', 'VERITA-QA'])]
order = ['High risk / low margin', 'Medium risk', 'Low risk / high margin']
for model_name in plot_df['Model'].unique():
    sub = plot_df[plot_df['Model'] == model_name].set_index('Risk stratum').reindex(order).reset_index()
    plt.plot(sub['Risk stratum'], sub['Accuracy'], marker='o', label=model_name)
plt.xticks(rotation=15, ha='right')
plt.ylabel('Accuracy')
plt.title('Risk-Stratified Accuracy After Calibration')
plt.legend(); plt.grid(alpha=0.3)
savefig('Fig7_Risk_Stratified_Accuracy_Calibrated.png')


In [ ]:
# ============================================================
# Cell 15b — Variational-equilibrium figures
# ============================================================
# Figure 8: Variational free energy and residual.
ve_plot = variational_equilibrium_df.sort_values('VE free energy')
x = np.arange(len(ve_plot))
plt.figure(figsize=(10,5))
plt.bar(x - 0.18, ve_plot['VE free energy'], width=0.36, label='VE free energy')
plt.bar(x + 0.18, ve_plot['Equilibrium residual'], width=0.36, label='Equilibrium residual')
plt.xticks(x, ve_plot['Model'], rotation=25, ha='right')
plt.ylabel('Variational-equilibrium score')
plt.title('Variational Equilibrium: Free Energy and Residual')
plt.legend(); plt.grid(axis='y', alpha=0.3)
savefig('Fig8_Variational_Equilibrium_Free_Energy.png')

# Figure 9: Stabilizing versus destabilizing forces.
plt.figure(figsize=(8,6))
for _, r in variational_equilibrium_df.iterrows():
    plt.scatter(r['Destabilizing force'], r['Stabilizing force'], s=80)
    plt.text(r['Destabilizing force'] + 0.01, r['Stabilizing force'], r['Model'], fontsize=8)
lo = min(variational_equilibrium_df['Destabilizing force'].min(), variational_equilibrium_df['Stabilizing force'].min())
hi = max(variational_equilibrium_df['Destabilizing force'].max(), variational_equilibrium_df['Stabilizing force'].max())
plt.plot([lo, hi], [lo, hi], linestyle='--', label='Force balance line')
plt.xlabel('Destabilizing force')
plt.ylabel('Stabilizing force')
plt.title('Variational Force-Balance Map')
plt.legend(); plt.grid(alpha=0.3)
savefig('Fig9_Variational_Force_Balance_Map.png')

# Figure 10: Accuracy, calibration safety, uncertainty safety, and action utility radar-style line view.
radar_terms = ['U_norm', 'C_norm', 'H_norm', 'G_norm', 'A_norm']
term_labels = ['Utility', 'Calibration safety', 'Uncertainty safety', 'Coverage', 'Action utility']
plt.figure(figsize=(10,5))
for _, r in variational_equilibrium_df.iterrows():
    plt.plot(term_labels, [r[t] for t in radar_terms], marker='o', label=r['Model'])
plt.ylim(0, 1.05)
plt.ylabel('Normalized equilibrium component')
plt.title('Normalized Variational-Equilibrium Components')
plt.xticks(rotation=15, ha='right')
plt.legend(fontsize=8); plt.grid(alpha=0.3)
savefig('Fig10_Variational_Equilibrium_Components.png')


## Dynamic variational-equilibrium upgrade

The previous cells estimate a static free-energy operating point. The following upgrade follows the roadmap by treating the retrieval-augmented decision process as a **dynamic decision field**. Each model starts from a perturbed operating state and relaxes toward an equilibrium attractor in the stabilizing/destabilizing force plane.

This cell implements three reviewer-relevant ideas:

1. **dynamic equilibrium trajectories** under nominal and perturbed conditions;
2. **phase-space visualization** of force convergence;
3. **local Lyapunov-style stability diagnostics**, using the decrease of a quadratic energy around each target operating point.

The purpose is not to overclaim physical exactness, but to provide a reproducible computational bridge between the static free-energy functional and the proposed interpretation of retrieval intelligence as a system seeking informational equilibrium.


In [ ]:
# ============================================================
# Cell 15c — Dynamic variational-equilibrium trajectories and stability
# ============================================================
# This cell implements the roadmap's dynamic-field upgrade without requiring a full re-run.
# It treats each model's static variational-equilibrium point as an attractor and simulates
# how a perturbed decision state relaxes toward that attractor under different deployment scenarios.

REQUIRED_VE_COLUMNS = [
    'Model', 'U_norm', 'C_norm', 'H_norm', 'R_norm', 'K_norm', 'G_norm', 'A_norm',
    'Stabilizing force', 'Destabilizing force', 'VE free energy'
]
missing_cols = [c for c in REQUIRED_VE_COLUMNS if c not in variational_equilibrium_df.columns]
if missing_cols:
    raise ValueError(f'Missing columns required for dynamic VE analysis: {missing_cols}')

def _clip01(x):
    return float(np.clip(x, 0.0, 1.0))

def _components_from_row(row):
    return {
        'U_norm': _clip01(row['U_norm']),
        'C_norm': _clip01(row['C_norm']),
        'H_norm': _clip01(row['H_norm']),
        'R_norm': _clip01(row['R_norm']),
        'K_norm': _clip01(row['K_norm']),
        'G_norm': _clip01(row['G_norm']),
        'A_norm': _clip01(row['A_norm']),
    }

def _scenario_components(base_components, scenario):
    c = dict(base_components)
    if scenario == 'Nominal relaxation':
        pass
    elif scenario == 'No calibration control':
        # Calibration safety collapses when the calibration force is removed.
        c['C_norm'] = 0.0
    elif scenario == 'Retrieval-noise perturbation':
        # Noisy retrieval reduces coverage and makes redundancy/complexity less safe.
        c['G_norm'] = _clip01(0.65 * c['G_norm'])
        c['R_norm'] = _clip01(0.70 * c['R_norm'])
        c['K_norm'] = _clip01(0.90 * c['K_norm'])
    elif scenario == 'Coverage-capped deployment':
        # Low-resource deployment caps the available evidence coverage.
        c['G_norm'] = min(c['G_norm'], 0.30)
    else:
        raise ValueError(f'Unknown scenario: {scenario}')
    return c

def _ve_forces_from_components(c):
    stabilizing = (
        VE_CONFIG['lambda_utility'] * c['U_norm'] +
        VE_CONFIG['lambda_calibration'] * c['C_norm'] +
        VE_CONFIG['lambda_uncertainty'] * c['H_norm'] +
        VE_CONFIG['lambda_redundancy'] * c['R_norm'] +
        VE_CONFIG['lambda_complexity'] * c['K_norm'] +
        VE_CONFIG['lambda_coverage'] * c['G_norm'] +
        VE_CONFIG['lambda_action'] * c['A_norm']
    )
    destabilizing = (
        VE_CONFIG['lambda_calibration'] * (1 - c['C_norm']) +
        VE_CONFIG['lambda_uncertainty'] * (1 - c['H_norm']) +
        VE_CONFIG['lambda_redundancy'] * (1 - c['R_norm']) +
        VE_CONFIG['lambda_complexity'] * (1 - c['K_norm'])
    )
    free_energy = (
        -VE_CONFIG['lambda_utility'] * c['U_norm'] +
        VE_CONFIG['lambda_calibration'] * (1 - c['C_norm']) +
        VE_CONFIG['lambda_uncertainty'] * (1 - c['H_norm']) +
        VE_CONFIG['lambda_redundancy'] * (1 - c['R_norm']) +
        VE_CONFIG['lambda_complexity'] * (1 - c['K_norm']) -
        VE_CONFIG['lambda_coverage'] * c['G_norm'] -
        VE_CONFIG['lambda_action'] * c['A_norm']
    )
    return float(stabilizing), float(destabilizing), float(free_energy)

def simulate_dynamic_equilibrium(row, scenario):
    cfg = DYNAMIC_CONFIG
    base_components = _components_from_row(row)
    components = _scenario_components(base_components, scenario)
    target_s, target_d, target_f = _ve_forces_from_components(components)

    # Deterministic perturbation: insufficient stabilization and inflated risk burden at t=0.
    s = max(0.0, 0.72 * target_s)
    d = max(0.0, 1.28 * target_d + 0.05)
    target = np.array([target_s, target_d], dtype=float)
    state = np.array([s, d], dtype=float)

    alpha = cfg['alpha_relaxation']
    beta = cfg['beta_coupling']
    gamma = cfg['gamma_utility']
    delta = cfg['delta_risk']
    dt = cfg['dt']

    # Continuous-time local Jacobian of the relaxation/coupling field:
    # d(state)/dt = alpha(target-state) - beta * [[1,-1],[-1,1]] state + controls
    jacobian = np.array([[-alpha - beta, beta], [beta, -alpha - beta]], dtype=float)
    eigvals = np.linalg.eigvals(jacobian)
    locally_stable = bool(np.all(np.real(eigvals) < 0))

    rows = []
    prev_v = None
    for t in range(cfg['n_steps'] + 1):
        lyapunov_v = float(np.sum((state - target) ** 2))
        field_potential = float(state[1] - state[0])
        rows.append({
            'Model': row['Model'],
            'Scenario': scenario,
            'Step': t,
            'Stabilizing force trajectory': float(state[0]),
            'Destabilizing force trajectory': float(state[1]),
            'Target stabilizing force': target_s,
            'Target destabilizing force': target_d,
            'Target VE free energy': target_f,
            'Field potential Phi': field_potential,
            'Lyapunov V': lyapunov_v,
            'Delta Lyapunov V': np.nan if prev_v is None else float(lyapunov_v - prev_v),
            'Jacobian eigenvalue 1': float(np.real(eigvals[0])),
            'Jacobian eigenvalue 2': float(np.real(eigvals[1])),
            'Locally stable': locally_stable,
        })
        prev_v = lyapunov_v

        # Discrete-time field update: relaxation + force-balance coupling + utility/risk control.
        coupling = np.array([-(state[0] - state[1]), -(state[1] - state[0])], dtype=float)
        control = np.array([gamma * components['U_norm'], -delta * (1 - components['C_norm'])], dtype=float)
        dstate = alpha * (target - state) + beta * coupling + control
        state = state + dt * dstate

    return rows

scenarios = [
    'Nominal relaxation',
    'No calibration control',
    'Retrieval-noise perturbation',
    'Coverage-capped deployment',
]

trajectory_rows = []
for _, row in variational_equilibrium_df.iterrows():
    for scenario in scenarios:
        trajectory_rows.extend(simulate_dynamic_equilibrium(row, scenario))

dynamic_trajectory_df = pd.DataFrame(trajectory_rows)
dynamic_trajectory_df.to_csv(TABLE_DIR / 'Table_8d_dynamic_equilibrium_trajectories.csv', index=False)

# Stability summary: final Lyapunov contraction and local Jacobian sign.
stability_rows = []
for (model, scenario), g in dynamic_trajectory_df.groupby(['Model', 'Scenario']):
    g = g.sort_values('Step')
    initial_v = float(g['Lyapunov V'].iloc[0])
    final_v = float(g['Lyapunov V'].iloc[-1])
    stability_rows.append({
        'Model': model,
        'Scenario': scenario,
        'Initial Lyapunov V': initial_v,
        'Final Lyapunov V': final_v,
        'Lyapunov contraction %': 100.0 * (initial_v - final_v) / max(initial_v, VE_CONFIG['eps']),
        'Max positive Delta V': float(g['Delta Lyapunov V'].dropna().max()) if g['Delta Lyapunov V'].notna().any() else np.nan,
        'Jacobian eigenvalue 1': float(g['Jacobian eigenvalue 1'].iloc[0]),
        'Jacobian eigenvalue 2': float(g['Jacobian eigenvalue 2'].iloc[0]),
        'Locally stable': bool(g['Locally stable'].iloc[0]),
    })

dynamic_stability_df = pd.DataFrame(stability_rows)
dynamic_stability_df.to_csv(TABLE_DIR / 'Table_8e_dynamic_equilibrium_stability.csv', index=False)

# Figure 11: dynamic trajectories in the force-balance plane.
plt.figure(figsize=(10, 7))
for (model, scenario), g in dynamic_trajectory_df.groupby(['Model', 'Scenario']):
    if scenario != 'Nominal relaxation':
        continue
    g = g.sort_values('Step')
    plt.plot(g['Destabilizing force trajectory'], g['Stabilizing force trajectory'], marker='o', markersize=3, label=model)
    plt.scatter(g['Target destabilizing force'].iloc[-1], g['Target stabilizing force'].iloc[-1], s=90, marker='x')
lo = min(dynamic_trajectory_df['Destabilizing force trajectory'].min(), dynamic_trajectory_df['Stabilizing force trajectory'].min())
hi = max(dynamic_trajectory_df['Destabilizing force trajectory'].max(), dynamic_trajectory_df['Stabilizing force trajectory'].max())
plt.plot([lo, hi], [lo, hi], linestyle='--', label='Force-balance line')
plt.xlabel('Destabilizing force trajectory')
plt.ylabel('Stabilizing force trajectory')
plt.title('Dynamic Variational-Equilibrium Trajectories')
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
savefig('Fig11_Dynamic_Equilibrium_Trajectories.png')

# Figure 12: Lyapunov contraction under perturbations.
plt.figure(figsize=(10, 6))
plot_df = dynamic_stability_df.copy()
plot_df['Label'] = plot_df['Model'] + '\n' + plot_df['Scenario'].str.replace(' ', '\n')
x = np.arange(len(plot_df))
plt.bar(x, plot_df['Lyapunov contraction %'])
plt.xticks(x, plot_df['Label'], rotation=90, fontsize=7)
plt.ylabel('Lyapunov contraction (%)')
plt.title('Dynamic Stability: Lyapunov Contraction Across Perturbation Scenarios')
plt.grid(axis='y', alpha=0.3)
savefig('Fig12_Lyapunov_Contraction.png')

display(dynamic_stability_df.sort_values(['Scenario', 'Lyapunov contraction %'], ascending=[True, False]).head(12))


In [ ]:
# ============================================================
# Cell 16 — Qualitative UVIF case studies
# ============================================================
case_rows = []
uvif_df = calibrated_results['VERITA-QA']
id_to_row = {r['id']: r for r in EVAL_ROWS}

# Examples where UVIF is correct and confidence is high.
for _, rec in uvif_df[uvif_df['correct'] == 1].sort_values('confidence', ascending=False).head(5).iterrows():
    r = id_to_row[rec['id']]
    evidence = ' '.join([corpus_texts[i] for i in rec['context_ids']])[:650]
    case_rows.append({
        'Question': r['question'],
        'Options': ' | '.join([f'{chr(65+i)}: {o}' for i, o in enumerate(r['options'])]),
        'Correct Answer': f'{chr(65+r["answer_idx"])}: {r["options"][r["answer_idx"]]}',
        'Predicted Answer': f'{chr(65+rec["pred"])}: {r["options"][rec["pred"]]}',
        'Confidence': rec['confidence'],
        'Entropy': rec['entropy'],
        'Top UVIF Evidence': evidence
    })
case_df = pd.DataFrame(case_rows)
case_df.to_csv(TABLE_DIR / 'Table_9_public_uvif_case_studies.csv', index=False)
case_df

In [ ]:
# ============================================================
# Cell 17 — Save outputs summary
# ============================================================
summary_lines = []
summary_lines.append('VERITA-QA Dynamic Variational-Equilibrium Public Dataset Notebook Outputs Summary')
summary_lines.append('='*72)
summary_lines.append(f'BASE_DIR: {BASE_DIR}')
summary_lines.append(f'Dataset: {CONFIG["dataset_name"]}')
summary_lines.append(f'Train/Validation/Test used: {len(train_rows)}/{len(val_rows)}/{len(test_rows)}')
summary_lines.append(f'Retrieval corpus passages: {len(corpus_df)}')
summary_lines.append('')
summary_lines.append('Runtime reductions:')
summary_lines.append(f'- Test questions: {CONFIG["max_eval_questions"]}')
summary_lines.append(f'- UVIF tuning questions: {CONFIG["max_uvif_tune_questions"]}')
summary_lines.append(f'- UVIF grid combinations: {len(list(grid_dicts(CONFIG["uvif_grid"])))}')
summary_lines.append(f'- Retrieval candidates: dense {CONFIG["top_k_dense"]}, sparse {CONFIG["top_k_sparse"]}, rerank budget {CONFIG["rerank_budget"]}')
summary_lines.append('')
summary_lines.append('Selected UVIF parameters:')
for k, v in BEST_UVIF_PARAMS.items():
    summary_lines.append(f'- {k}: {v}')
summary_lines.append('')
summary_lines.append('Main held-out test results before calibration:')
for _, row in performance_df.iterrows():
    summary_lines.append(f'- {row["Model"]}: Accuracy={row["Accuracy"]:.4f}, Macro-F1={row["Macro-F1"]:.4f}, ECE={row["ECE"]:.4f}')
summary_lines.append('')
summary_lines.append('Pre/post calibration summary:')
for _, row in calibration_comparison_df.iterrows():
    summary_lines.append(f'- {row["Model"]}: T={row["Selected_temperature"]:.2f}, ECE before={row["ECE_before"]:.4f}, ECE after={row["ECE_after"]:.4f}, relative ECE reduction={row["Relative_ECE_reduction_%"]:.1f}%')
summary_lines.append('')
summary_lines.append('Variational-equilibrium operating points:')
for _, row in variational_equilibrium_df.iterrows():
    summary_lines.append(f'- {row["Model"]}: VE free energy={row["VE free energy"]:.4f}, residual={row["Equilibrium residual"]:.4f}, VE rank={int(row["VE rank"])}')
summary_lines.append('')
if 'dynamic_stability_df' in globals():
    summary_lines.append('Dynamic variational-equilibrium stability:')
    for _, row in dynamic_stability_df.iterrows():
        summary_lines.append(f'- {row["Model"]} | {row["Scenario"]}: Lyapunov contraction={row["Lyapunov contraction %"]:.2f}%, locally stable={row["Locally stable"]}')

summary_lines.append('')
summary_lines.append('Bootstrap comparison:')
summary_lines.append(boot_df.to_string(index=False))
summary_lines.append('')
summary_lines.append('Saved tables:')
for p in sorted(TABLE_DIR.glob('*.csv')):
    summary_lines.append(f'- {p.name}')
summary_lines.append('')
summary_lines.append('Saved figures:')
for p in sorted(FIG_DIR.glob('*.png')):
    summary_lines.append(f'- {p.name}')
summary_lines.append('')
summary_lines.append('Interpretation note:')
summary_lines.append('This upgraded notebook is intended to support a VERITA-QA variational-equilibrium article. Claims should emphasize equilibrium among utility, calibration, uncertainty, evidence stability, complexity, and selective-action reliability rather than forcing raw accuracy superiority.')

summary_path = OUTPUT_DIR / 'outputs_summary_verita_qa_dynamic_variational_equilibrium.txt'
summary_path.write_text('\\n'.join(summary_lines), encoding='utf-8')


## Manuscript-use guidance

This upgraded notebook is best used for a new article focused on **UVIF as a variational-equilibrium layer**. The safest scientific claim is not raw accuracy dominance, but a more defensible equilibrium claim:

> UVIF introduces a measurable variational-control layer for retrieval-augmented educational QA, balancing predictive utility, calibration, uncertainty, redundancy, coverage, complexity, and selective-action efficiency. On public SciQ evaluation, the notebook assesses whether this equilibrium improves reliability-oriented behavior under a transparent CPU-reproducible setting.

The notebook now provides explicit equilibrium tables and figures that can support a methodology section, results subsection, and discussion on equilibrium residuals, force balance, and operational decision stability.


# Final Scientific Positioning

This notebook supports **VERITA-QA: Variational Equilibrium Retrieval Intelligence for Trustworthy Answering**.

The key claim is not that VERITA-QA must always dominate raw accuracy. Rather, VERITA-QA provides a **controllable equilibrium layer** for retrieval-augmented intelligence, where model behavior is interpreted through utility--risk balance, calibration and uncertainty stabilization, evidence redundancy versus coverage, selective-action reliability, equilibrium residuals, dynamic trajectories, and force-balance diagnostics.

The framework should therefore be presented as **equilibrium-centric AI**, not as another accuracy-centric RAG benchmark.
